# **Acquiring real-world data**



> Web scraping, web harvesting, or web data extraction is data scraping used for extracting data from websites.[1] Web scraping software may directly access the World Wide Web using the Hypertext Transfer Protocol or a web browser. While web scraping can be done manually by a software user, the term typically refers to automated processes implemented using a bot or web crawler. It is a form of copying in which specific data is gathered and copied from the web, typically into a central local database or spreadsheet, for later retrieval or analysis. (https://en.wikipedia.org/wiki/Web_scraping)

To access data on the internet, some level of familiarity with basic protocolls and website structures is very beneficial. Python provides means of communicating with websites and automatizing the data collection process.



---


# **HTTP**



> The Hypertext Transfer Protocol (HTTP) is designed to enable communications between clients and servers. HTTP works as a request-response protocol between a client and server.

> Example: A client (browser) sends an HTTP request to the server; then the server returns a response to the client. The response contains status information about the request and may also contain the requested content. (https://www.w3schools.com/tags/ref_httpmethods.asp)

There are multiple HTTP Methods that implement different behavior in communications between clients and servers, but we will focus on **GET** and (to lesser extend) **POST**:

https://www.w3schools.com/tags/ref_httpmethods.asp

A GET request is as easy as copying a URL (like the one above) to your browser's adress field. The resulting server response is the website that you are trying to access. Parameters/Data are submitted via the URL.

**Example: Search functionality on Uni Trier homepage**

    https://www.uni-trier.de/suche?id=62&q=computerlinguistik

If instead, we would be searching for "Natural Language Processing", the resulting URL would look like this:

    https://www.uni-trier.de/suche?id=62&q=natural+language+processing

In this example, using the search field yields the same results as manipulating the URL directly.





The *requests* library allows to send http requests and receive server responses. (https://requests.readthedocs.io/en/latest/)

In [ ]:
import requests

r = requests.get("https://www.uni-trier.de/suche?id=62&q=natural+language+processing")
r

Without further modification, the request only displays the status code of the response (https://www.w3schools.com/tags/ref_httpmessages.asp). To access the contents of the website, the *.text* attribute can be used. Now the unformatted .html sourcecode of the website is displayed.

In [ ]:
r.text



---

# **HTML - Scraping with BeautifulSoup**
>  
    - HTML stands for Hyper Text Markup Language
    - HTML is the standard markup language for creating Web pages
    - HTML describes the structure of a Web page
    - HTML consists of a series of elements
    - HTML elements tell the browser how to display the content
    - HTML elements label pieces of content such as "this is a heading", "this is a paragraph", "this is a link", etc.
https://www.w3schools.com/Html/html_intro.asp


The output above shows the raw, unformatted .html source of the accessed website. The HTML structure of a website can be used to access only specific elements.

Below is an example of a simple HTML page:

    <!DOCTYPE html>
    <html>
    <head>
      <title>Page Title</title>
    </head>
    <body>

      <h1>My First Heading</h1>
      <p>My first paragraph.</p>

    </body>
    </html>

By using a HTML parser, elements can be accessed by their names. HTML can be represented as a String in Python, which can then be parse via a library like *BeautifulSoup* (https://www.crummy.com/software/BeautifulSoup/bs4/doc/):

In [ ]:
import bs4 #BeautifulSoup

htmlPage = "<!DOCTYPE html>\
<html>\
<head>\
  <title>Page Title</title>\
</head>\
<body>\
  <h1>My First Heading</h1>\
  <p>My first paragraph.</p>\
  <p>My second paragraph.</p>\
  <p>My third paragraph.</p>\
  <p>My fourth paragraph.</p>\
</body>\
</html>" #HTML page

soup = bs4.BeautifulSoup(htmlPage, "html.parser") #parse HTML page

In [ ]:
soup.h1 #access first <h1> element

In [ ]:
soup.p #access first p element

In [ ]:
soup.find_all("p") #access all p elements

Elements are ordered in a hierachical, tree-like structure, with parents, siblings, etc.

In [ ]:
#access the parent element of p -> body element
soup.p.parent

In [ ]:
soup.p.parent.parent

In [ ]:
soup.p.find_next_sibling() #access the next sibling of p

Elements can not only contain values (i.e. texts), but can also have attributes:

    <a href="https://www.w3schools.com">Visit W3Schools</a>

The *\<a> \</a>* element denotes a hyperlink, that should contain a href attribute which denotes the link adress. The element content itself is the displayed text of the hyperlink:

    <a href="https://www.w3schools.com">Visit W3Schools</a>

<a href="https://www.w3schools.com">Visit W3Schools</a>

By having a look a the HTML code, it is possible to determine how the desired data are stored. In the next step, by using a HTML parser, the elements can automatically be retrieved and used for further analysis.



---
# **Example application: How many study programmes are there at Trier University?**

A list of available bachelor programs can be found here: https://www.uni-trier.de/studium/studienangebot/bachelor

Accordingly, available master programs can be found under this link: https://www.uni-trier.de/studium/studienangebot/master

The goals are:
- Determine how many study programs are there
- Determine the distribution of different degrees across the programs (Bachelore of Science, etc.)

**Step 1:**
Have a look at the website and determine which information you need
https://www.uni-trier.de/studium/studienangebot/bachelor

**Step 2:**
Find the corresponding representation in the source code view-source:https://www.uni-trier.de/studium/studienangebot/bachelor

**Step 3:**
Find the common elements or element properties that contain your data

In [ ]:
#call the page and find all relevant elements (in this case: study programs)
pageBachelors = bs4.BeautifulSoup(requests.get("https://www.uni-trier.de/studium/studienangebot/bachelor").text, "html.parser") #visit page
#find all <a title=*> elements with href starting "/studium/studienangebot/studienfaecher/studiengang"

In [ ]:
studyPrograms = pageBachelors.find_all("a", attrs={"title": True}, href=lambda x: x and x.startswith("/studium/studienangebot/studienfaecher/studiengang"))
studyPrograms[0]

In [ ]:
#add the master's programs
pageMasters = bs4.BeautifulSoup(requests.get("https://www.uni-trier.de/studium/studienangebot/master").text, "html.parser")
studyProgramsComplete = studyPrograms + pageMasters.find_all("a", attrs={"title": True}, href=lambda x: x and x.startswith("/studium/studienangebot/studienfaecher/studiengang"))
len(studyProgramsComplete)

In [ ]:
studyProgramsComplete

Selecting only certain attributes can be achieved by referencing the attribute name:

In [ ]:
studyProgramsComplete[0].attrs #show available attributes

In [ ]:
#only display the title attribute
studyProgramsComplete[0]['title']
#for element in studyProgramsComplete:
#  print(element["title"])

In [ ]:
#display only the content of the element
studyProgramsComplete[0].text.strip()
#for element in studyProgramsComplete:
#  print(element.text.strip())

In [ ]:
#how many Hauptfach, Nebenfach and 1-Fach are there?
degreeCount = {"Hauptfach": 0, "Nebenfach": 0, "1-Fach": 0}
for element in studyProgramsComplete:
  if "Hauptfach" in element.text:
    degreeCount["Hauptfach"] += 1
  elif "Nebenfach" in element.text:
    degreeCount["Nebenfach"] += 1
  elif "1-Fach" in element.text:
    degreeCount["1-Fach"] += 1
degreeCount

# **How are the obtainable degrees distributed (Bachelor of Science, Master of Arts, etc.)?**

For this, multiple pages need to be visited. The first page contains a number of hyperlinks, which can can be accessed and analyzed further.

Again, the first step is to have a look at the page and its html source:

https://www.uni-trier.de/studium/studienangebot/studienfaecher/studiengang?sgaid=350&cHash=200ddfe669832a2a5f9758bbcc7ee08d

A quick search (Control + F) reveals that the desired information is contained in an element called:

    <div class="steckbrief bg-blue">

The element contains a table defined by:

    <table class="padding-left-1 padding-right-1">

And the table contains a row with the seeked information:

    <tr><td class="label">Abschluss</td><td class="value">Master of Arts</td></tr>



In [ ]:
programPage = bs4.BeautifulSoup(requests.get("https://www.uni-trier.de/studium/studienangebot/studienfaecher/studiengang?sgaid=350&cHash=200ddfe669832a2a5f9758bbcc7ee08d").text, "html.parser")
#find all tr with "Abschluss"
abschluss = programPage.find("td", string="Abschluss")
#find their sibling
abschluss.find_next_sibling().text

In order to get the information for all programs at Trier University, it is required to iterate over the urls from the first example, visit every page, and collect the information:

In [ ]:
import time

baseUrl = "https://www.uni-trier.de/" #this needs to be combined with the href
print (baseUrl + studyProgramsComplete[0]["href"]) #example
countDict = {} #empty dictionary to save the counts

for element in studyProgramsComplete: #iterate over all elements
  currentURL = baseUrl + element['href'] #build the current url
  currentPage = bs4.BeautifulSoup(requests.get(currentURL).text, "html.parser") #visit the page
  time.sleep(0.2) #take a little break to prevent getting banned
  abschluss = currentPage.find("td", string="Abschluss").find_next_sibling().text #find the desired element
  if abschluss in countDict: #do the counting
    countDict[abschluss] += 1
  else:
    countDict[abschluss] = 1
print (countDict)

In [ ]:
#sum the program count from the dict
sum(countDict.values())



---



# **API access with requests**
according to Wikipedia:

> An application programming interface (API) is a connection between computers or between computer programs. It is a type of software interface, offering a service to other pieces of software. A document or standard that describes how to build such a connection or interface is called an API specification. A computer system that meets this standard is said to implement or expose an API. The term API may refer either to the specification or to the implementation.

The *requests* library natively supports interactions with APIs. Many websites deploy APIs as a way of using their services.

**Example:**

The German Bundestag allows for retrieving protocolls of its sessions via an API. The usage is documented under the following links:
- https://dip.bundestag.de/%C3%BCber-dip/hilfe/api#content
- https://search.dip.bundestag.de/api/v1/swagger-ui/#/


**Authentification**
All requests require a valid API Key to be passed:

In [ ]:
import requests

baseUrl = "https://search.dip.bundestag.de/api/v1/"
apiKey = "lolNo" #false key

payload = {'apikey': apiKey}

r = requests.get('https://search.dip.bundestag.de/api/v1/vorgang', params=payload) #returns 401 - authenticfication required, as the provided key is not correct
r

In [ ]:
import requests

baseUrl = "https://search.dip.bundestag.de/api/v1/"
apiKey = "OSOegLs.PR2lwJ1dwCeje9vTj7FPOt3hvpYKtwKkhw" #taken from their website

parameters = {'apikey': apiKey}

r = requests.get('https://search.dip.bundestag.de/api/v1/plenarprotokoll-text/1', params=parameters) #getting the protocoll by id
r

In [ ]:
r.text

In [ ]:
type(r.text)

In [ ]:
import json

protocol = json.loads(r.text) #removing the newline characters and loading the string as json
protocol['text']